# Multimodal Medical Report Generation — ViT + BioGPT (v4)

**Goal:** Automatically generate structured radiology reports from chest X-ray images.

**Architecture:** ViT (Vision Encoder) → Cross-Attention Fusion → BioGPT (Text Decoder)

**New in v4 (over v3):**
- Cross-attention fusion layer (text attends to image features before generation)
- METEOR evaluation metric added alongside BLEU/ROUGE
- Better report formatting (handles abbreviations correctly)
- Generated reports saved as CSV for easy analysis
- `weights_only=True` in `torch.load()` for PyTorch 2.x compatibility
- Updated per-module LR groups for new cross-attention layer

**Carried from v3:**
- Fixed image normalization bug, early stopping, per-module LR
- Gradient checkpointing, error handling, Kaggle-compatible

## 1. Kaggle Environment Setup

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# timm, transformers, nltk are pre-installed on Kaggle — do NOT reinstall them
# (reinstalling triggers hours-long dependency resolution)
# Only install the small packages that are missing:
!pip install -q --no-deps rouge-score sacremoses

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('Setup complete.')

## 2. Imports & Configuration

In [ ]:
import os
import sys
import csv
import json
import math
import time
import random
import shutil
import re
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

try:
    from torch.amp import autocast, GradScaler
    AMP_NEW_API = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    AMP_NEW_API = False

from transformers import (
    ViTModel,
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

print(f'PyTorch version: {torch.__version__}')
print(f'AMP API: {"new (torch.amp)" if AMP_NEW_API else "legacy (torch.cuda.amp)"}')
print('All imports successful.')

In [ ]:
# ======================== CONFIGURATION ========================

# Data paths (Kaggle-specific)
MERGED_JSONL = Path('/kaggle/input/merged-dataset/merged_dataset.jsonl')
MODEL_SAVE_DIR = Path('/kaggle/working/models/vit_biogpt_multimodal_v4')
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Model paths (Kaggle model inputs)
VIT_PATH = '/kaggle/input/vit2/transformers/default/1'
BIOGPT_PATH = '/kaggle/input/biogpt2/pytorch/default/1'
TOKENIZER_PATH = '/kaggle/input/bio-gpt/pytorch/default/1'

# Image
IMAGE_SIZE = 224
IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

# Training hyperparameters
MAX_SAMPLES = 15000
BATCH_SIZE = 1
ACCUM_STEPS = 8
EPOCHS = 10
WEIGHT_DECAY = 0.01
MAX_GEN_TOKENS = 256
MAX_SEQ_LEN = 512
SEED = 42
NUM_WORKERS = 0
PIN_MEMORY = False

# Per-module learning rates
LR_VIT = 1e-5
LR_PROJ = 5e-4        # projection + cross-attention (randomly initialized)
LR_GPT = 2e-5

# Cross-attention config
CROSS_ATTN_HEADS = 8
CROSS_ATTN_DROPOUT = 0.1

# Early stopping
PATIENCE = 3

# Generation parameters
NUM_BEAMS = 4
LENGTH_PENALTY = 1.2
REPETITION_PENALTY = 2.5
NO_REPEAT_NGRAM = 3
MIN_GEN_TOKENS = 50

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'Compute capability: {props.major}.{props.minor}')

In [ ]:
def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed()

## 3. Utility Functions

In [ ]:
def fix_path(win_path: str) -> str:
    """Convert Windows-style image paths from JSONL to Kaggle Linux paths."""
    path = win_path.replace('\\', '/')
    marker = 'official_data_iccv_final/files/'
    idx = path.find(marker)
    if idx == -1:
        filename = os.path.basename(path)
        return f'/kaggle/input/mimic-cxr-dataset/official_data_iccv_final/files/{filename}'
    else:
        return f'/kaggle/input/mimic-cxr-dataset/{path[idx:]}'


def preprocess_image(image_path: str, image_size: int = IMAGE_SIZE) -> torch.Tensor:
    """Load & normalize image for ViT. Returns [1, 3, H, W] tensor."""
    mean = np.array(IMAGE_MEAN, dtype='float32').reshape(1, 1, 3)
    std = np.array(IMAGE_STD, dtype='float32').reshape(1, 1, 3)

    if image_path.lower().endswith('.npz'):
        d = np.load(image_path, allow_pickle=True)
        pix = torch.from_numpy(d['pixel'].astype('float32'))
        if pix.shape[1] != image_size or pix.shape[2] != image_size:
            pix = nn.functional.interpolate(
                pix.unsqueeze(0), size=(image_size, image_size),
                mode='bilinear', align_corners=False
            ).squeeze(0)
        return pix.unsqueeze(0)
    else:
        im = Image.open(image_path).convert('RGB').resize(
            (image_size, image_size), resample=Image.BILINEAR
        )
        arr = np.array(im).astype('float32') / 255.0
        arr = (arr - mean) / std
        arr = np.transpose(arr, (2, 0, 1))
        return torch.from_numpy(arr).unsqueeze(0).to(torch.float32)


def get_autocast_context(enabled=True):
    if AMP_NEW_API:
        return autocast('cuda', enabled=enabled and device.type == 'cuda')
    else:
        return autocast(enabled=enabled and device.type == 'cuda')


def get_grad_scaler():
    if AMP_NEW_API:
        return GradScaler('cuda')
    else:
        return GradScaler()


def smart_sentence_split(text):
    """Split text into sentences, handling abbreviations like Dr., No., etc."""
    abbreviations = r'(?<!Dr)(?<!Mr)(?<!Mrs)(?<!Ms)(?<!No)(?<!vs)(?<!etc)(?<!i\.e)(?<!e\.g)'
    sentences = re.split(abbreviations + r'\.\s+', text)
    return [s.strip() for s in sentences if s.strip()]


print('Utilities defined.')

## 4. Dataset

In [ ]:
class MedicalReportDataset(Dataset):
    """Dataset for chest X-ray images paired with radiology reports."""

    def __init__(self, jsonl_path, tokenizer, image_size=IMAGE_SIZE,
                 max_len=MAX_SEQ_LEN, max_samples=MAX_SAMPLES):
        self.data = []
        with open(jsonl_path, encoding='utf8') as f:
            for line in f:
                self.data.append(json.loads(line))
                if max_samples and len(self.data) >= max_samples:
                    break
        self.tokenizer = tokenizer
        self.image_size = image_size
        self.max_len = max_len
        self.mean = np.array(IMAGE_MEAN, dtype='float32').reshape(1, 1, 3)
        self.std = np.array(IMAGE_STD, dtype='float32').reshape(1, 1, 3)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        rec = self.data[idx]
        img_path = fix_path(rec.get('image_path', ''))

        try:
            im = Image.open(img_path).convert('RGB').resize(
                (self.image_size, self.image_size), resample=Image.BILINEAR
            )
            arr = np.array(im).astype('float32') / 255.0
            arr = (arr - self.mean) / self.std
            arr = np.transpose(arr, (2, 0, 1))
            pixel_tensor = torch.from_numpy(arr).to(torch.float32)
        except Exception as e:
            print(f'[WARNING] Failed to load image {idx}: {img_path} — {e}')
            pixel_tensor = torch.zeros(3, self.image_size, self.image_size, dtype=torch.float32)

        report = str(rec.get('target_report', '')).strip()
        prefix = 'Patient history: \nReport: '
        full_text = prefix + report

        enc = self.tokenizer(
            full_text, truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt',
        )
        input_ids = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)

        labels = input_ids.clone()
        pref_enc = self.tokenizer(
            prefix, truncation=True, max_length=64, return_tensors='pt'
        )['input_ids'].squeeze(0)
        pref_len = (pref_enc != self.tokenizer.pad_token_id).sum().item()
        labels[:pref_len] = -100

        return {
            'pixel_values': pixel_tensor,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }


print('Dataset class defined.')

## 5. Model Architecture

**New: Cross-Attention Fusion Layer**

Instead of simply concatenating image and text embeddings, text tokens now **attend to image features** via multi-head cross-attention before being fed to BioGPT. This gives the language model richer, more targeted access to visual information.

```
ViT → Projection → [Image Embeddings]
                         ↓ (key, value)
                    Cross-Attention ← Text Embeddings (query)
                         ↓
                    Fused Text Embeddings
                         ↓
            [Image Emb] + [Fused Text Emb] → BioGPT → Report
```

In [ ]:
class CrossAttentionFusion(nn.Module):
    """Cross-attention layer where text attends to image features."""

    def __init__(self, hidden_size, num_heads=CROSS_ATTN_HEADS,
                 dropout=CROSS_ATTN_DROPOUT):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            hidden_size, num_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 4, hidden_size),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, text_emb, img_emb):
        attn_out, _ = self.cross_attn(
            query=text_emb, key=img_emb, value=img_emb
        )
        text_emb = self.norm1(text_emb + attn_out)
        ffn_out = self.ffn(text_emb)
        text_emb = self.norm2(text_emb + ffn_out)
        return text_emb


class BioGPTMultimodal(nn.Module):
    def __init__(self, vit_model, gpt_model):
        super().__init__()
        self.vit = vit_model
        self.gpt = gpt_model
        gpt_hidden = gpt_model.config.hidden_size

        self.proj = nn.Linear(vit_model.config.hidden_size, gpt_hidden)
        self.cross_attn = CrossAttentionFusion(
            gpt_hidden, num_heads=CROSS_ATTN_HEADS, dropout=CROSS_ATTN_DROPOUT
        )

    def enable_gradient_checkpointing(self):
        if hasattr(self.vit, 'gradient_checkpointing_enable'):
            self.vit.gradient_checkpointing_enable()
        if hasattr(self.gpt, 'gradient_checkpointing_enable'):
            self.gpt.gradient_checkpointing_enable()

    def forward(self, pixel_values, input_ids, attention_mask, labels=None):
        vit_out = self.vit(pixel_values=pixel_values).last_hidden_state
        img_emb = self.proj(vit_out)

        token_emb = self.gpt.get_input_embeddings()(input_ids)
        fused_text = self.cross_attn(token_emb, img_emb)

        inputs_embeds = torch.cat([img_emb, fused_text], dim=1)

        img_mask = torch.ones(
            (input_ids.size(0), img_emb.size(1)),
            dtype=attention_mask.dtype, device=attention_mask.device,
        )
        new_attn = torch.cat([img_mask, attention_mask], dim=1)

        if labels is not None:
            pad = torch.full(
                (labels.size(0), img_emb.size(1)), -100,
                dtype=labels.dtype, device=labels.device,
            )
            new_labels = torch.cat([pad, labels], dim=1)
        else:
            new_labels = None

        out = self.gpt(
            inputs_embeds=inputs_embeds,
            attention_mask=new_attn,
            labels=new_labels,
            return_dict=True,
        )
        return out


print('Model classes defined (with CrossAttentionFusion).')

## 6. Load Pretrained Components & Build Model

In [ ]:
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})

print('Loading ViT...')
vit = ViTModel.from_pretrained(VIT_PATH)

print('Loading BioGPT...')
biogpt = AutoModelForCausalLM.from_pretrained(BIOGPT_PATH)
biogpt.resize_token_embeddings(len(tokenizer))

model = BioGPTMultimodal(vit, biogpt).to(device)
model.enable_gradient_checkpointing()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
cross_attn_params = sum(p.numel() for p in model.cross_attn.parameters())
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Cross-attention parameters: {cross_attn_params:,} (new in v4)')

## 7. Prepare Data Loaders

In [ ]:
dataset = MedicalReportDataset(
    MERGED_JSONL, tokenizer,
    image_size=IMAGE_SIZE, max_len=MAX_SEQ_LEN, max_samples=MAX_SAMPLES,
)

n = len(dataset)
if n == 0:
    raise SystemExit('Dataset empty — check merged_dataset.jsonl paths.')

train_n = int(0.9 * n)
val_n = n - train_n
train_ds, val_ds = random_split(
    dataset, [train_n, val_n],
    generator=torch.Generator().manual_seed(SEED),
)
print(f'Train: {len(train_ds)} | Validation: {len(val_ds)}')


def collate_fn(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]).to(device),
        'input_ids': torch.stack([b['input_ids'] for b in batch]).to(device),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]).to(device),
        'labels': torch.stack([b['labels'] for b in batch]).to(device),
    }


train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)
print('Data loaders ready.')

## 8. Optimizer, Scheduler & Training Setup

Per-module LRs: ViT (1e-5) | Projection + CrossAttn (5e-4) | BioGPT (2e-5)

In [ ]:
no_decay = ['bias', 'LayerNorm.weight']

def get_param_groups(model):
    groups = []
    # ViT
    groups.append({'params': [p for n, p in model.vit.named_parameters()
                              if not any(nd in n for nd in no_decay)],
                   'lr': LR_VIT, 'weight_decay': WEIGHT_DECAY})
    groups.append({'params': [p for n, p in model.vit.named_parameters()
                              if any(nd in n for nd in no_decay)],
                   'lr': LR_VIT, 'weight_decay': 0.0})
    # Projection layer
    groups.append({'params': [p for n, p in model.proj.named_parameters()
                              if not any(nd in n for nd in no_decay)],
                   'lr': LR_PROJ, 'weight_decay': WEIGHT_DECAY})
    groups.append({'params': [p for n, p in model.proj.named_parameters()
                              if any(nd in n for nd in no_decay)],
                   'lr': LR_PROJ, 'weight_decay': 0.0})
    # Cross-attention fusion (same LR as projection — randomly initialized)
    groups.append({'params': [p for n, p in model.cross_attn.named_parameters()
                              if not any(nd in n for nd in no_decay)],
                   'lr': LR_PROJ, 'weight_decay': WEIGHT_DECAY})
    groups.append({'params': [p for n, p in model.cross_attn.named_parameters()
                              if any(nd in n for nd in no_decay)],
                   'lr': LR_PROJ, 'weight_decay': 0.0})
    # BioGPT
    groups.append({'params': [p for n, p in model.gpt.named_parameters()
                              if not any(nd in n for nd in no_decay)],
                   'lr': LR_GPT, 'weight_decay': WEIGHT_DECAY})
    groups.append({'params': [p for n, p in model.gpt.named_parameters()
                              if any(nd in n for nd in no_decay)],
                   'lr': LR_GPT, 'weight_decay': 0.0})
    return groups


optimizer = torch.optim.AdamW(get_param_groups(model))
total_steps = math.ceil(len(train_loader) * EPOCHS / ACCUM_STEPS)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.05 * total_steps)),
    num_training_steps=total_steps,
)
scaler = get_grad_scaler()

print(f'Total optimization steps: {total_steps}')
print(f'LR: ViT={LR_VIT}, Projection+CrossAttn={LR_PROJ}, BioGPT={LR_GPT}')

## 9. Training Loop (with Early Stopping)

In [ ]:
best_val = float('inf')
best_path = MODEL_SAVE_DIR / 'best_multimodal_v4.pt'
patience_counter = 0
training_log = []

print(f'Starting training: {EPOCHS} epochs, early stopping patience = {PATIENCE}')
print('=' * 70)

global_step = 0

for epoch in range(EPOCHS):
    print(f"\n{'='*70}")
    print(f'EPOCH {epoch + 1}/{EPOCHS}')
    print(f"{'='*70}")

    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f'Train {epoch+1}/{EPOCHS}')
    for step, batch in pbar:
        with get_autocast_context():
            out = model(
                batch['pixel_values'], input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'], labels=batch['labels'],
            )
            loss = out.loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

        pbar.set_postfix({
            'Batch': f'{loss.item() * ACCUM_STEPS:.4f}',
            'Avg': f'{running_loss / (step + 1):.4f}',
        })

    avg_train = running_loss / len(train_loader)

    print('Validating...')
    model.eval()
    val_loss = 0.0
    v_steps = 0

    val_pbar = tqdm(val_loader, desc=f'Val {epoch+1}/{EPOCHS}')
    with torch.no_grad():
        for batch in val_pbar:
            with get_autocast_context():
                out = model(
                    batch['pixel_values'], input_ids=batch['input_ids'],
                    attention_mask=batch['attention_mask'], labels=batch['labels'],
                )
                val_loss += out.loss.item()
                v_steps += 1
            val_pbar.set_postfix({
                'Batch': f'{out.loss.item():.4f}',
                'Avg': f'{val_loss / v_steps:.4f}',
            })

    avg_val = val_loss / v_steps if v_steps > 0 else float('inf')

    epoch_log = {
        'epoch': epoch + 1,
        'train_loss': round(avg_train, 4),
        'val_loss': round(avg_val, 4),
        'best_val': round(min(best_val, avg_val), 4),
    }
    training_log.append(epoch_log)
    print(f'\nEpoch {epoch+1}  |  Train: {avg_train:.4f}  |  Val: {avg_val:.4f}')

    if avg_val < best_val:
        best_val = avg_val
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict()}, best_path)
        tokenizer.save_pretrained(MODEL_SAVE_DIR)
        vit.save_pretrained(MODEL_SAVE_DIR / 'vit')
        biogpt.save_pretrained(MODEL_SAVE_DIR / 'biogpt')
        print(f'Saved best model -> {best_path}')
    else:
        patience_counter += 1
        print(f'No improvement ({patience_counter}/{PATIENCE})')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping triggered at epoch {epoch + 1}!')
        break

print(f"\n{'='*70}")
print(f'Training complete. Best val loss: {best_val:.4f}')
print(f"{'='*70}")

## 10. Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = [d['epoch'] for d in training_log]
train_losses = [d['train_loss'] for d in training_log]
val_losses = [d['val_loss'] for d in training_log]

plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, train_losses, 'bo-', label='Train Loss')
plt.plot(epochs_ran, val_losses, 'ro-', label='Validation Loss')

best_epoch = min(training_log, key=lambda x: x['val_loss'])['epoch']
best_loss = min(training_log, key=lambda x: x['val_loss'])['val_loss']
plt.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7,
            label=f'Best (epoch {best_epoch})')
plt.scatter([best_epoch], [best_loss], color='green', s=100, zorder=5)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss (v4 — with Cross-Attention)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(MODEL_SAVE_DIR / 'training_curve.png'), dpi=150)
plt.show()

with open(MODEL_SAVE_DIR / 'training_log.json', 'w') as f:
    json.dump(training_log, f, indent=2)
print('Training log saved.')

## 11. Load Best Checkpoint

In [ ]:
if best_path.exists():
    checkpoint = torch.load(best_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f'Loaded best checkpoint from {best_path}')
else:
    print('No checkpoint found — using last training state.')
    model.eval()

## 12. Report Generation Functions

In [ ]:
def generate_raw_report(image_path, history_text='',
                        max_new_tokens=MAX_GEN_TOKENS, num_beams=NUM_BEAMS):
    """Generate raw report text from an X-ray image."""
    model.eval()
    pix = preprocess_image(image_path, IMAGE_SIZE).to(device)

    prompt = f'Patient history: {history_text}\nReport: '
    prefix_enc = tokenizer(prompt, return_tensors='pt').to(device)
    prefix_ids = prefix_enc['input_ids']
    prefix_attn = prefix_enc['attention_mask']

    with torch.no_grad():
        vit_out = model.vit(pixel_values=pix).last_hidden_state
        img_emb = model.proj(vit_out)
        token_emb = model.gpt.get_input_embeddings()(prefix_ids)

        fused_text = model.cross_attn(token_emb, img_emb)

        inputs_embeds = torch.cat([img_emb, fused_text], dim=1)
        img_mask = torch.ones((1, img_emb.size(1)),
                              dtype=prefix_attn.dtype).to(device)
        new_mask = torch.cat([img_mask, prefix_attn], dim=1).to(device)

        gen_ids = model.gpt.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=new_mask,
            max_new_tokens=max_new_tokens,
            min_new_tokens=MIN_GEN_TOKENS,
            num_beams=num_beams,
            length_penalty=LENGTH_PENALTY,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
        )

        out_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
        if prompt in out_text:
            out_text = out_text.split(prompt, 1)[1]

    return out_text.strip()


print('Report generation function defined.')

In [ ]:
def format_structured_report(raw_text, patient_info=None):
    """
    Post-process raw model output into a structured medical report.
    """
    raw_text = raw_text.replace('_ _ _', '[DATE]').replace('_ _', '[NAME]').strip()

    findings = ''
    impression = ''
    raw_lower = raw_text.lower()

    findings_markers = ['findings:', 'finding:']
    impression_markers = ['impression:', 'impressions:']

    for marker in findings_markers:
        idx = raw_lower.find(marker)
        if idx != -1:
            text_after = raw_text[idx + len(marker):].strip()
            imp_idx = -1
            for imp_marker in impression_markers:
                imp_search = text_after.lower().find(imp_marker)
                if imp_search != -1:
                    imp_idx = imp_search
                    break
            if imp_idx != -1:
                findings = text_after[:imp_idx].strip()
                impression = text_after[imp_idx:].strip()
                for imp_marker in impression_markers:
                    if impression.lower().startswith(imp_marker):
                        impression = impression[len(imp_marker):].strip()
                        break
            else:
                findings = text_after.strip()
            break

    if not findings and not impression:
        for imp_marker in impression_markers:
            if raw_lower.startswith(imp_marker):
                impression = raw_text[len(imp_marker):].strip()
                break
        if not impression:
            findings = raw_text

    sep = '=' * 60
    dash = '-' * 60
    lines = []
    lines.append(sep)
    lines.append('           CHEST X-RAY RADIOLOGY REPORT')
    lines.append(sep)

    if patient_info:
        lines.append('')
        lines.append('PATIENT INFORMATION:')
        lines.append(f'  {patient_info}')

    if findings:
        lines.append('')
        lines.append(dash)
        lines.append('FINDINGS:')
        lines.append(dash)
        sentences = smart_sentence_split(findings)
        for i, s in enumerate(sentences, 1):
            s = s.strip()
            if s and not s.endswith('.'):
                s += '.'
            if s:
                lines.append(f'  {i}. {s}')

    if impression:
        lines.append('')
        lines.append(dash)
        lines.append('IMPRESSION:')
        lines.append(dash)
        sentences = smart_sentence_split(impression)
        for i, s in enumerate(sentences, 1):
            s = s.strip()
            if s and not s.endswith('.'):
                s += '.'
            if s:
                lines.append(f'  * {s}')

    lines.append('')
    lines.append(dash)
    lines.append('RECOMMENDATIONS:')
    lines.append(dash)
    lines.append('  * Clinical correlation is recommended.')
    lines.append('  * Follow-up imaging may be warranted based on clinical context.')
    lines.append('  * Consult attending physician for treatment decisions.')

    lines.append('')
    lines.append(dash)
    lines.append('DISCLAIMER:')
    lines.append(dash)
    lines.append('  This report was generated by an AI model (ViT + BioGPT).')
    lines.append('  It is NOT a substitute for professional medical diagnosis.')
    lines.append('  Always consult a qualified radiologist for clinical decisions.')
    lines.append('')
    lines.append(sep)

    return '\n'.join(lines)


print('Report formatter defined.')

## 13. Sample Report Generation

In [ ]:
with open(MERGED_JSONL, 'r', encoding='utf8') as f:
    sample = json.loads(next(f))

sample_img = fix_path(sample['image_path'])
history = '55-year-old female with cough and mild fever.'

print(f'Image: {sample_img}')
print(f'History: {history}')
print()

raw = generate_raw_report(sample_img, history_text=history, max_new_tokens=200)
print('Raw model output:')
print(raw)
print()

structured = format_structured_report(raw, patient_info=history)
print(structured)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 14))
ax.axis('off')
plt.text(
    0.02, 0.98, structured,
    fontsize=11, fontfamily='monospace', verticalalignment='top',
    transform=ax.transAxes, wrap=True,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8),
)
plt.savefig(str(MODEL_SAVE_DIR / 'sample_report.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Report saved to {MODEL_SAVE_DIR / "sample_report.png"}')

## 14. Evaluation — BLEU, ROUGE & METEOR Scores

**New:** METEOR metric added — considers synonyms and stemming, better suited for medical text.

In [ ]:
def evaluate_metrics(model, val_dataset, tokenizer, num_samples=100):
    """
    Compute BLEU, ROUGE, and METEOR scores on a subset of the validation set.
    """
    model.eval()
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smooth = SmoothingFunction().method1

    bleu_scores = {1: [], 2: [], 3: [], 4: []}
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    meteor_scores = []

    indices = list(range(len(val_dataset)))
    if len(indices) > num_samples:
        random.shuffle(indices)
        indices = indices[:num_samples]

    print(f'\nEvaluating on {len(indices)} validation samples...')

    for i, idx in enumerate(tqdm(indices, desc='Evaluating')):
        original_idx = val_dataset.indices[idx]
        rec = val_dataset.dataset.data[original_idx]

        reference = str(rec.get('target_report', '')).strip()
        if not reference:
            continue

        img_path = fix_path(rec.get('image_path', ''))

        try:
            predicted = generate_raw_report(
                img_path, history_text='',
                max_new_tokens=200, num_beams=NUM_BEAMS,
            )
        except Exception as e:
            print(f'  [Skip] Error for sample {idx}: {e}')
            continue

        if not predicted.strip():
            continue

        ref_tokens = reference.lower().split()
        pred_tokens = predicted.lower().split()

        for n in range(1, 5):
            weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
            try:
                score = sentence_bleu(
                    [ref_tokens], pred_tokens,
                    weights=weights, smoothing_function=smooth,
                )
                bleu_scores[n].append(score)
            except Exception:
                pass

        rouge_result = rouge.score(reference, predicted)
        for key in rouge_scores:
            rouge_scores[key].append(rouge_result[key].fmeasure)

        try:
            m_score = meteor_score([ref_tokens], pred_tokens)
            meteor_scores.append(m_score)
        except Exception:
            pass

    results = {}
    for n in range(1, 5):
        results[f'BLEU-{n}'] = float(np.mean(bleu_scores[n])) if bleu_scores[n] else 0.0
    for key in rouge_scores:
        results[key.upper()] = float(np.mean(rouge_scores[key])) if rouge_scores[key] else 0.0
    results['METEOR'] = float(np.mean(meteor_scores)) if meteor_scores else 0.0

    return results


print('Evaluation function defined (BLEU + ROUGE + METEOR).')

In [ ]:
metrics = evaluate_metrics(model, val_ds, tokenizer, num_samples=100)

print('\n' + '=' * 50)
print('       EVALUATION RESULTS (v4)')
print('=' * 50)
for metric, value in metrics.items():
    print(f'  {metric:12s}: {value:.4f}')
print('=' * 50)

with open(MODEL_SAVE_DIR / 'eval_metrics.json', 'w') as f:
    json.dump({k: round(v, 4) for k, v in metrics.items()}, f, indent=2)
print(f'\nMetrics saved to {MODEL_SAVE_DIR / "eval_metrics.json"}')

## 15. Generate Multiple Sample Reports & Save as CSV

In [ ]:
NUM_REPORT_SAMPLES = 10
print(f'Generating {NUM_REPORT_SAMPLES} sample reports from validation set...\n')

sample_indices = random.sample(range(len(val_ds)), min(NUM_REPORT_SAMPLES, len(val_ds)))
csv_rows = []

for i, idx in enumerate(sample_indices):
    original_idx = val_ds.indices[idx]
    rec = val_ds.dataset.data[original_idx]
    img_path = fix_path(rec.get('image_path', ''))
    reference = str(rec.get('target_report', '')).strip()

    print(f"{'='*70}")
    print(f'SAMPLE {i+1}: {os.path.basename(img_path)}')
    print(f"{'='*70}")

    try:
        predicted = generate_raw_report(img_path, history_text='', max_new_tokens=200)
        structured = format_structured_report(predicted)

        print('\nREFERENCE REPORT:')
        print(f'  {reference[:300]}...' if len(reference) > 300 else f'  {reference}')
        print('\nGENERATED REPORT:')
        print(structured)

        csv_rows.append({
            'image': os.path.basename(img_path),
            'reference': reference,
            'generated': predicted,
        })
    except Exception as e:
        print(f'  [ERROR] {e}')

    print()

csv_path = MODEL_SAVE_DIR / 'generated_reports.csv'
with open(csv_path, 'w', newline='', encoding='utf8') as f:
    writer = csv.DictWriter(f, fieldnames=['image', 'reference', 'generated'])
    writer.writeheader()
    writer.writerows(csv_rows)
print(f'\nGenerated reports saved to {csv_path}')

## 16. Metrics Comparison Visualization

In [ ]:
import matplotlib.pyplot as plt

metric_names = list(metrics.keys())
metric_values = list(metrics.values())

colors = ['#2196F3', '#1976D2', '#1565C0', '#0D47A1',
          '#4CAF50', '#388E3C', '#2E7D32', '#FF9800']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(metric_names, metric_values, color=colors[:len(metric_names)],
              edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Score', fontsize=13)
ax.set_title('Evaluation Metrics — v4 (with Cross-Attention + METEOR)', fontsize=14)
ax.set_ylim(0, max(metric_values) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(MODEL_SAVE_DIR / 'metrics_chart.png'), dpi=150)
plt.show()
print('Metrics chart saved.')

## 17. Summary

| Component | Details |
|-----------|--------|
| Vision Encoder | ViT-Small (patch16, 224x224) |
| Language Model | BioGPT (347M params) |
| Bridge | Linear projection (768 -> 1024) + **Cross-Attention Fusion** |
| Dataset | MIMIC-CXR (15K samples, 90/10 split) |
| Training | Mixed precision, gradient checkpointing, per-module LR |
| Regularization | Early stopping (patience=3), weight decay, grad clipping |
| Evaluation | BLEU-1/2/3/4 + ROUGE-1/2/L + **METEOR** |
| Generation | Beam search with length and repetition penalties |
| Output | Structured report: Findings, Impression, Recommendations |
| Export | Reports saved as CSV, metrics as JSON, plots as PNG |